# Resolving a coraplex designator with a language model

`llm_reasoner` reflects a query into typed, path-named slots and depends only on `krrood`.
World knowledge reaches it through two seams implemented in `coraplex.querying.language_model`,
which is where the dependency on `semantic_digital_twin` lives:

```
krrood  <-  llm_reasoner  <-  coraplex.querying.language_model  ->  semantic_digital_twin
```

This notebook resolves a real `PickUpAction` against a real world, with no model in the loop --
a `ScriptedClient` replays fixed responses.

In [1]:
import json

from krrood.entity_query_language.factories import a, variable

from coraplex.datastructures.enums import Arms
from coraplex.datastructures.grasp import GraspDescription
from coraplex.querying.language_model import (
    SituationCandidateRenderer,
    SituationContextProvider,
)
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from llm_reasoner.clients import ScriptedClient
from llm_reasoner.resolver import (
    LanguageModelBackend,
    NoValidCandidates,
    reflect_slots,
)

## The world

An apartment with milk, cereal and a spoon, plus a PR2 rooted under `map -> odom_combined`.

**The apartment matters.** A robot-only world contains nothing manipulable, so every world-derived
section of the prompt is correctly empty and neither the target description nor the permitted
grasps can be exercised. `REPOSITORY_ROOT` resolves two levels up from `coraplex/doc/`.

In [2]:
import os

from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.robots.robot_parts import AbstractRobot, EndEffector
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
    Milk,
    Spoon,
)
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.connections import (
    Connection6DoF,
    FixedConnection,
)
from semantic_digital_twin.world_description.world_entity import Body

from coraplex.querying.language_model import end_effectors_of

REPOSITORY_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
RESOURCES = os.path.join(REPOSITORY_ROOT, "coraplex", "resources")


def build_robot_world(robot_semantic_annotation=PR2) -> World:
    """Parse a robot from URDF and root it under map -> odom_combined."""
    world = URDFParser.from_file(
        file_path=robot_semantic_annotation.get_ros_file_path()
    ).parse()
    robot_semantic_annotation.from_world(world)

    with world.modify_world():
        old_root = world.root
        map_body = Body(name=PrefixedName("map"))
        localization_body = Body(name=PrefixedName("odom_combined"))
        world.add_connection(
            Connection6DoF.create_with_dofs(world, map_body, localization_body)
        )
        drive = robot_semantic_annotation.get_drive_connection_type().create_with_dofs(
            parent=localization_body, child=old_root, world=world
        )
        world.add_connection(drive)
        drive.has_hardware_interface = True

    return world


def build_apartment_world() -> World:
    """Parse the apartment and place annotated objects in it."""
    apartment = URDFParser.from_file(
        os.path.join(RESOURCES, "worlds", "apartment.urdf")
    ).parse()
    milk = STLParser(os.path.join(RESOURCES, "objects", "milk.stl")).parse()
    cereal = STLParser(
        os.path.join(RESOURCES, "objects", "breakfast_cereal.stl")
    ).parse()
    spoon = STLParser(os.path.join(RESOURCES, "objects", "spoon.stl")).parse()

    apartment.merge_world_at_pose(
        milk,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 2, 1.05, reference_frame=apartment.root
        ),
    )
    apartment.merge_world_at_pose(
        cereal,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 2.5, 1.05, reference_frame=apartment.root
        ),
    )
    apartment.merge_world(
        spoon,
        FixedConnection(
            parent=apartment.get_body_by_name("cabinet10_drawer_top"),
            child=spoon.root,
            parent_T_connection_expression=HomogeneousTransformationMatrix.from_xyz_rpy(
                -0.05, -0.05, -0.02
            ),
        ),
    )

    with apartment.modify_world():
        apartment.add_semantic_annotations(
            [
                Milk(root=apartment.get_body_by_name("milk.stl")),
                Spoon(root=apartment.get_body_by_name("spoon.stl")),
            ]
        )
        apartment.add_semantic_annotation_recursively(
            Drawer(
                root=apartment.get_body_by_name("cabinet10_drawer_top"),
                handle=Handle(root=apartment.get_body_by_name("handle_cab10_t")),
            )
        )

    return apartment


world = build_apartment_world()
world.merge_world(build_robot_world(PR2))

# Stand the robot right next to the milk instead of at the apartment origin.
# test_designator/test_action_conditions.py::test_pick_up_post_condition proves
# this exact placement reachable and runs a full pickup against this same milk
# position; pr2_apartment_world's own (1.3, 2, 0) is only a starting pose and
# is NOT close enough -- test_pick_up_pre_conditions asserts it still fails.
world.get_body_by_name("base_footprint").parent_connection.origin = (
    HomogeneousTransformationMatrix.from_xyz_rpy(1.8, 2, 0)
)

robot = world.get_semantic_annotations_by_type(AbstractRobot)[0]
end_effectors = end_effectors_of(robot)
milk = world.get_body_by_name("milk.stl")
milk_annotation = world.get_semantic_annotations_by_type(Milk)[0]

print(robot.name.name, "->", [e.name.name for e in end_effectors])
print("target:", milk.name.name)

Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]


PR2 -> ['PR2LeftGripper', 'PR2RightGripper']
target: milk.stl


## The query

`PickUpAction` is a good specimen because it mixes every slot kind at once:

| slot | kind | why |
|---|---|---|
| `object_designator` | `BOUND` | the milk, given by the caller |
| `arm` | `VALUE` | a two-member enum; the answer depends on world state |
| `tolerate_grasp_stall` | `VALUE` | a boolean judgement |
| `grasp_description.*` | nested | five slots including an individual to select |

Binding `object_designator` directly is what lets the bridge identify the target without guessing
from coordinates. Conditions bind through `match.variable`; `match.expression` compiles,
verbalizes correctly, and then matches nothing.

In [3]:
def underspecified_pick_up():
    match = a(PickUpAction)(
        object_designator=milk_annotation,
        arm=...,
        tolerate_grasp_stall=...,
        grasp_description=a(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            end_effector=variable(EndEffector, end_effectors),
            rotate_gripper=...,
            manipulation_offset=...,
        ),
    )
    match.where(match.variable.grasp_description.manipulation_offset > 0.0)
    return match


for slot in reflect_slots(underspecified_pick_up()):
    detail = ""
    if slot.members:
        detail = " | ".join(member.name for member in slot.members)
    elif slot.domain is not None:
        detail = f"{len(slot.domain)} candidate(s)"
    print(f"{slot.kind.name:9} {slot.path:48} {slot.type_name:18} {detail}")

BOUND     PickUpAction.object_designator                   Body               
VALUE     PickUpAction.arm                                 Arms               LEFT | RIGHT | BOTH
VALUE     PickUpAction.tolerate_grasp_stall                bool               
VALUE     PickUpAction.grasp_description.approach_direction ApproachDirection  FRONT | BACK | RIGHT | LEFT
VALUE     PickUpAction.grasp_description.vertical_alignment VerticalAlignment  NoAlignment | TOP | BOTTOM
REFERENT  PickUpAction.grasp_description.end_effector      EndEffector        2 candidate(s)
VALUE     PickUpAction.grasp_description.rotate_gripper    bool               
VALUE     PickUpAction.grasp_description.manipulation_offset float              


Docstrings come straight out of `coraplex`. Nothing was written twice -- the prompt's account
of the task is the source code.

## The prompt

In [4]:
backend = LanguageModelBackend(
    client=ScriptedClient(),
    context_provider=SituationContextProvider(world=world, robot=robot),
    candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
)
print(backend.describe(underspecified_pick_up()))

Generate 3 candidate PickUpAction object(s).

## What these are

**PickUpAction** -- Let the robot pick up an object.

**GraspDescription** -- Describes a grasp configuration for a end_effector the description consists of the
approach direction (the side from which to grasp e.g. FRONT, LEFT, etc and the
vertical alignment (TOP, BOTTOM).

## The request

Generate a PickUpAction whose object_designator is a specific Body, and whose arm is Ellipsis, such that it has tolerate_grasp_stall is Ellipsis, the approach_direction of its grasp_description is Ellipsis, the vertical_alignment of its grasp_description is Ellipsis, the end_effector of its grasp_description is a JustinLeftHand, a JustinRightHand, a PR2RightGripper, or a PR2LeftGripper, its grasp_description has rotate_gripper is Ellipsis, the manipulation_offset of its grasp_description is Ellipsis, and the manipulation_offset of its grasp_description is greater than 0.0

## Situation

The object being acted on:
- milk.stl  a Milk  0.0

### What the world contributes

Four things, none derivable from the schema:

1. **The target**, identified from the bound `object_designator` -- its sortal annotations, size,
   what supports it, and where it lies *relative to the robot* rather than in world coordinates.
2. **Permitted grasps**, computed by coraplex's own
   `GraspDescription.calculate_grasp_descriptions`. Approach directions name faces of the
   *object*, so without this the model would be choosing a face of a frame it cannot see.
3. **End effector occupancy**, and which side of the robot each is mounted on.
4. **Other objects** in the scene.

Pass `robot=` to the renderer or the candidate lines fall back to name and occupancy only.

In [5]:
provider = SituationContextProvider(world=world, robot=robot)
print(provider.context_for(underspecified_pick_up()))

The object being acted on:
- milk.stl  a Milk  0.06 x 0.07 x 0.19 m (depth x width x height)  on apartment_root  0.57 m ahead, 1.05 m above of base_footprint

Grasps permitted by how milk.stl is turned relative to the robot.
Approach directions name faces of the object, not of the robot, and the first listed is the face most turned toward the robot:
- approach_direction FRONT, vertical_alignment NoAlignment
- approach_direction LEFT, vertical_alignment NoAlignment
- approach_direction FRONT, vertical_alignment TOP
- approach_direction LEFT, vertical_alignment TOP
- approach_direction FRONT, vertical_alignment BOTTOM
- approach_direction LEFT, vertical_alignment BOTTOM

Positions below are given relative to PR2, which faces along its own forward axis.
Its end effectors:
- PR2LeftGripper: free
- PR2RightGripper: free

Other objects in the scene:
- spoon.stl  a Spoon  0.22 x 0.05 x 0.02 m (depth x width x height)  on cabinet10_drawer_top  0.66 m ahead, 0.20 m to the left, 0.81 m above of 

In [6]:
renderer = SituationCandidateRenderer(world=world, robot=robot)
match = underspecified_pick_up()
for end_effector in end_effectors:
    print(renderer.render(end_effector, match))

PR2LeftGripper  free  mounted on the robot's left  must span 0.07 m to grasp milk.stl
PR2RightGripper  free  mounted on the robot's right  must span 0.07 m to grasp milk.stl


Turn the sections off individually to see what each contributes:

In [7]:
bare = SituationContextProvider(
    world=world,
    robot=robot,
    describe_target=False,
    describe_grasps=False,
    describe_scene=False,
)
print(repr(bare.context_for(underspecified_pick_up())))

'Positions below are given relative to PR2, which faces along its own forward axis.\nIts end effectors:\n- PR2LeftGripper: free\n- PR2RightGripper: free'


## Resolving

Write-back binds each mapped variable by its access path, pushes literals into the nested kwargs,
then calls the factory the caller supplied. That factory is why `llm_reasoner` never imports the
classes it constructs.

In [8]:
def response(arm="LEFT", end_effector_token=0, manipulation_offset=0.06):
    return json.dumps([
        {
            "PickUpAction.arm": arm,
            "PickUpAction.tolerate_grasp_stall": False,
            "PickUpAction.grasp_description.approach_direction": "FRONT",
            "PickUpAction.grasp_description.vertical_alignment": "TOP",
            "PickUpAction.grasp_description.end_effector": end_effector_token,
            "PickUpAction.grasp_description.rotate_gripper": False,
            "PickUpAction.grasp_description.manipulation_offset": manipulation_offset,
        }
    ])


def resolve(raw):
    return LanguageModelBackend(
        client=ScriptedClient(responses=[raw]),
        context_provider=SituationContextProvider(world=world, robot=robot),
        candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
        number_of_candidates=1,
    ).evaluate(underspecified_pick_up())


action = next(iter(resolve(response(end_effector_token=1))))

print(action.arm)
print(action.grasp_description.approach_direction)
print("end effector:", action.grasp_description.end_effector.name.name)
print("is the live world object:", action.grasp_description.end_effector is end_effectors[1])
print("object:", action.object_designator.name.name)

LEFT
ApproachDirection.FRONT
end effector: PR2RightGripper
is the live world object: True
object: milk.stl


## Enforcement

A model is not a sound generator, so the where-conditions are used twice: as prompt content to
steer, and afterwards as a filter to enforce. The filter is krrood's own selective evaluation.

In [9]:
try:
    list(resolve(response(manipulation_offset=-0.02)))
except NoValidCandidates as error:
    print("rejected:", error)

rejected: All 1 constructed candidate(s) failed the query's where-conditions.


## A real model

Nothing above changes. Swap the client.

```bash
pip install anthropic
export ANTHROPIC_API_KEY=...
```

In [10]:
from dotenv import load_dotenv
load_dotenv(override=True)

from llm_reasoner.clients import AnthropicClient

live = LanguageModelBackend(
    client=AnthropicClient(),
    context_provider=SituationContextProvider(world=world, robot=robot),
    candidate_renderer=SituationCandidateRenderer(world=world, robot=robot),
    number_of_candidates=3,
)

for action in live.evaluate(underspecified_pick_up()):
    print(action.arm, action.grasp_description.approach_direction)

print(live.log[-1].raw_response)

RIGHT ApproachDirection.FRONT
LEFT ApproachDirection.LEFT
RIGHT ApproachDirection.FRONT
```json
[
  {
    "PickUpAction.arm": "RIGHT",
    "PickUpAction.tolerate_grasp_stall": false,
    "PickUpAction.grasp_description.approach_direction": "FRONT",
    "PickUpAction.grasp_description.vertical_alignment": "TOP",
    "PickUpAction.grasp_description.end_effector": 1,
    "PickUpAction.grasp_description.rotate_gripper": false,
    "PickUpAction.grasp_description.manipulation_offset": 0.08
  },
  {
    "PickUpAction.arm": "LEFT",
    "PickUpAction.tolerate_grasp_stall": true,
    "PickUpAction.grasp_description.approach_direction": "LEFT",
    "PickUpAction.grasp_description.vertical_alignment": "TOP",
    "PickUpAction.grasp_description.end_effector": 0,
    "PickUpAction.grasp_description.rotate_gripper": true,
    "PickUpAction.grasp_description.manipulation_offset": 0.06
  },
  {
    "PickUpAction.arm": "RIGHT",
    "PickUpAction.tolerate_grasp_stall": true,
    "PickUpAction.grasp_desc

## The experiment worth running

`arm` is the sharpest test in the library: thirteen actions take it, it has two members, and the
right answer depends entirely on world state the schema cannot carry.

Occupy one gripper, place the target clearly to one side, and score the model's choice over
several trials against a 50% baseline. The situation section reports occupancy and which side
each gripper is mounted on, and the target section says which side the object is on -- so if the
model does not beat chance, the context is not carrying its weight and that is worth knowing.

## Executing the resolved action

Same pattern as `nested_language_model_backend.ipynb`: start a ROS node for TF and marker publication, wrap the world and robot in a `Context`, resolve one action with the real client, and run it through the plan executor.

The robot was placed next to the milk when the world was built, close enough that `test_pick_up_post_condition` proves a pickup reachable from there. `ParkArmsAction` still runs first -- `PickUpAction`'s `IsObjectReachableBy` precondition plans a real IK/motion path, and the URDF's raw default arm configuration is not a reliable seed for that; parking puts both arms in a known-good pose to plan from.

In [11]:
import threading

import rclpy
from rclpy.node import Node
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

from coraplex.datastructures.dataclasses import Context


def start_ros_node(node_name: str) -> tuple[Node, threading.Thread]:
    """Start a ROS node and retain its background spinning thread."""
    if not rclpy.ok():
        rclpy.init()
    ros_node = rclpy.create_node(node_name)
    ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
    ros_thread.start()
    return ros_node, ros_thread


def start_visualization_publisher(
    world: World,
    ros_node: Node,
) -> tuple[TFPublisher, VizMarkerPublisher]:
    """Start synchronized transform and marker publication for a world."""
    tf_publisher = TFPublisher(_world=world, node=ros_node)
    visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)
    return tf_publisher, visualization_publisher


ros_node, ros_thread = start_ros_node("pick_up_action")
context = Context(world=world, robot=robot, ros_node=ros_node)



In [12]:
tf_publisher, visualization_publisher = start_visualization_publisher(world, ros_node)
print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")

RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


Resolve one action with the real client. `live` already exists from the cell above; calling `evaluate` again asks a fresh completion and returns a fresh iterator.

In [13]:
resolved_action = live.evaluate(underspecified_pick_up())
resolved_action = next(iter(resolved_action))
type(resolved_action)

coraplex.robot_plans.actions.core.pick_up.PickUpAction

### Execute the resolved action as part of the plan

In [14]:
from coraplex.execution_environment import simulated_robot
from coraplex.plans.factories import sequential
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from semantic_digital_twin.datastructures.definitions import TorsoState

task = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        resolved_action,
    ],
    context=context,
)

with simulated_robot:
    task.perform()